In [1]:
# CELL 1: FD001 – Q-MetaBoost

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
from scipy.ndimage import gaussian_filter1d
import warnings
import sys
import os
import time

warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True)

print("CELL 1: FD001 – Data Preparation")

# CONFIGURATION
RANDOM_SEED = 42
RUL_CAP = 100
SEQ_LEN = 30

# Q-MetaBoost hyperparameters
xgb_params = {
    'n_estimators': 1000,
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'verbosity': 0
}

SIGMA = 0.3


# 1. LOAD DATA
IF_KAGGLE = 'kaggle' in sys.modules or os.path.exists('/kaggle/input')
if IF_KAGGLE:
    data_path = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/'
else:
    data_path = './data/CMaps/'

col_names = ['ID', 'Cycle'] + [f'Op{i}' for i in range(1,4)] + [f'Sensor{i}' for i in range(1,22)]

train_df = pd.read_csv(data_path + 'train_FD001.txt', sep=r'\s+', header=None, names=col_names)
test_df = pd.read_csv(data_path + 'test_FD001.txt', sep=r'\s+', header=None, names=col_names)
rul_df = pd.read_csv(data_path + 'RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL'])

# Remove constant sensors
sensor_cols = [f'Sensor{i}' for i in range(1,22)]
constant = [c for c in sensor_cols if train_df[c].std() < 1e-6]
train_df.drop(columns=constant, inplace=True)
test_df.drop(columns=constant, inplace=True)
print(f"Removed constant sensors: {constant}")

# Add RUL (piecewise cap 100)
train_df['RUL'] = train_df.groupby('ID')['Cycle'].transform(lambda x: x.max() - x)
train_df['RUL'] = np.minimum(train_df['RUL'], RUL_CAP)

test_out = []
for eid in test_df['ID'].unique():
    temp = test_df[test_df['ID'] == eid].copy()
    last_rul = rul_df.iloc[eid-1, 0]
    temp['RUL'] = last_rul + (temp['Cycle'].max() - temp['Cycle'])
    test_out.append(temp)
test_df = pd.concat(test_out)
test_df['RUL'] = np.minimum(test_df['RUL'], RUL_CAP)


# 2. ADD ROLLING FEATURES (first 8 active sensors)
active_sensors = [c for c in train_df.columns if c.startswith('Sensor')]
sensors_for_rolling = active_sensors[:8]
print(f"Sensors for rolling: {sensors_for_rolling}")

def add_rolling_features(df, cols):
    df = df.copy()
    for c in cols:
        df[f'{c}_mean'] = df.groupby('ID')[c].transform(
            lambda x: x.expanding().mean().shift(1).fillna(x.iloc[0]))
        df[f'{c}_std'] = df.groupby('ID')[c].transform(
            lambda x: x.expanding().std().shift(1).fillna(0))
        df[f'{c}_rate'] = df.groupby('ID')[c].transform(
            lambda x: x.diff().fillna(0))
    return df

train_df = add_rolling_features(train_df, sensors_for_rolling)
test_df = add_rolling_features(test_df, sensors_for_rolling)

# Base features: active sensors + ops + rolling additions
op_cols = ['Op1', 'Op2', 'Op3']
rolling_cols = []
for c in sensors_for_rolling:
    rolling_cols.extend([f'{c}_mean', f'{c}_std', f'{c}_rate'])
base_features = active_sensors + op_cols + rolling_cols
print(f"Total base features: {len(base_features)}")   # 42


# 3. TRAIN/VALIDATION SPLIT (80/20 by engine ID)
ids = train_df['ID'].unique()
np.random.seed(RANDOM_SEED)
np.random.shuffle(ids)
split = int(0.8 * len(ids))
train_ids = ids[:split]
val_ids = ids[split:]

train_data = train_df[train_df['ID'].isin(train_ids)]
val_data   = train_df[train_df['ID'].isin(val_ids)]

np.save('fd001_train_ids.npy', train_ids)
np.save('fd001_val_ids.npy', val_ids)

X_train_raw = train_data[base_features].values
y_train_raw = train_data['RUL'].values
X_val_raw   = val_data[base_features].values
y_val_raw   = val_data['RUL'].values
X_test_raw  = test_df[base_features].values
y_test_raw  = test_df['RUL'].values


# 4. NORMALIZATION (standard scaling)
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_train = imputer.fit_transform(X_train_raw)
X_val   = imputer.transform(X_val_raw)
X_test  = imputer.transform(X_test_raw)

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


# 5. CREATE SEQUENCES (30-cycle windows)
def create_sequences(data, ids, targets, seq_len=SEQ_LEN):
    X_seq, y_seq = [], []
    for eid in np.unique(ids):
        mask = ids == eid
        edata = data[mask]
        etargets = targets[mask]
        for i in range(len(edata) - seq_len):
            X_seq.append(edata[i:i+seq_len])
            y_seq.append(etargets[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train, train_data['ID'].values, y_train_raw)
X_val_seq,   y_val_seq   = create_sequences(X_val,   val_data['ID'].values,   y_val_raw)
X_test_seq,  y_test_seq  = create_sequences(X_test,  test_df['ID'].values,    y_test_raw)

print(f"Train sequences: {X_train_seq.shape}, Val: {X_val_seq.shape}, Test: {X_test_seq.shape}")


# 6. FEATURE EXTRACTION (3,6,9,14 features per base)
def extract_features_vectorised(seq_data, n_features_per_base):
    n_windows, seq_len, n_base = seq_data.shape
    x = np.arange(seq_len, dtype=float)
    A_lin = np.column_stack([x, np.ones(seq_len)])
    A_quad = np.column_stack([x**2, x, np.ones(seq_len)])
    pinv_lin = np.linalg.pinv(A_lin)
    pinv_quad = np.linalg.pinv(A_quad)

    windows_by_sensor = seq_data.transpose(2, 0, 1)  # (n_base, n_windows, seq_len)
    all_feats = []
    for s_idx in range(n_base):
        w = windows_by_sensor[s_idx]                # (n_windows, seq_len)
        mean_vals = np.mean(w, axis=1)
        std_vals  = np.std(w, axis=1)
        range_vals = np.ptp(w, axis=1)
        stats_feats = np.column_stack([mean_vals, std_vals, range_vals])

        if n_features_per_base >= 6:
            lin_coeff = pinv_lin @ w.T
            slope = lin_coeff[0]
            intercept = lin_coeff[1]
            y_pred_lin = (A_lin @ lin_coeff).T
            ss_res = np.sum((w - y_pred_lin)**2, axis=1)
            ss_tot = np.sum((w - np.mean(w, axis=1, keepdims=True))**2, axis=1)
            r2 = 1 - ss_res / (ss_tot + 1e-12)
            lin_feats = np.column_stack([slope, intercept, r2])

        if n_features_per_base >= 9:
            quad_coeff = pinv_quad @ w.T
            beta2 = quad_coeff[0]
            beta1 = quad_coeff[1]
            quad_int = quad_coeff[2]
            quad_feats = np.column_stack([beta2, beta1, quad_int])

        if n_features_per_base == 14:
            momentum = np.mean(w[:, -5:], axis=1) - np.mean(w[:, :5], axis=1)
            mid = seq_len // 2
            roc_mid_start = w[:, mid] - w[:, 0]
            roc_end_mid   = w[:, -1] - w[:, mid]
            roc_avg       = (w[:, -1] - w[:, 0]) / seq_len
            roc_diff_mean = np.mean(np.diff(w, axis=1), axis=1)
            rate_feats = np.column_stack([roc_mid_start, roc_end_mid, roc_avg, roc_diff_mean])

        if n_features_per_base == 3:
            sensor_feats = stats_feats
        elif n_features_per_base == 6:
            sensor_feats = np.hstack([stats_feats, lin_feats])
        elif n_features_per_base == 9:
            sensor_feats = np.hstack([stats_feats, lin_feats, quad_feats])
        else:  # 14
            sensor_feats = np.hstack([stats_feats, lin_feats, quad_feats, momentum[:, None], rate_feats])

        all_feats.append(sensor_feats)

    return np.hstack(all_feats)

print("\nExtracting feature matrices")
start_time = time.time()

X_train_levels = {}
X_val_levels = {}
X_test_levels = {}
for nf in [3, 6, 9, 14]:
    print(f"Extracting {nf} features per base...")
    X_train_levels[nf] = extract_features_vectorised(X_train_seq, nf)
    X_val_levels[nf]   = extract_features_vectorised(X_val_seq, nf)
    X_test_levels[nf]  = extract_features_vectorised(X_test_seq, nf)
    scaler_feat = StandardScaler()
    X_train_levels[nf] = scaler_feat.fit_transform(X_train_levels[nf])
    X_val_levels[nf]   = scaler_feat.transform(X_val_levels[nf])
    X_test_levels[nf]  = scaler_feat.transform(X_test_levels[nf])

print(f"Feature extraction completed in {time.time()-start_time:.2f} seconds")

# Save full level (14 features)
np.save('fd001_X_train_full.npy', X_train_levels[14])
np.save('fd001_X_val_full.npy', X_val_levels[14])
np.save('fd001_X_test_full.npy', X_test_levels[14])
np.save('fd001_y_train.npy', y_train_seq)
np.save('fd001_y_val.npy', y_val_seq)
np.save('fd001_y_test.npy', y_test_seq)

# Save intermediate levels
for nf in [3, 6, 9]:
    np.save(f'fd001_X_train_{nf}.npy', X_train_levels[nf])
    np.save(f'fd001_X_val_{nf}.npy', X_val_levels[nf])
    np.save(f'fd001_X_test_{nf}.npy', X_test_levels[nf])


# 7. RAW BASELINE (A0: only active sensors + ops)
print("Creating raw sensor baseline (A0)")
raw_base_features = active_sensors + op_cols
imputer_raw = SimpleImputer(strategy='mean')
scaler_raw = StandardScaler()
X_train_raw_std = imputer_raw.fit_transform(train_data[raw_base_features].values)
X_val_raw_std   = imputer_raw.transform(val_data[raw_base_features].values)
X_test_raw_std  = imputer_raw.transform(test_df[raw_base_features].values)
X_train_raw_std = scaler_raw.fit_transform(X_train_raw_std)
X_val_raw_std   = scaler_raw.transform(X_val_raw_std)
X_test_raw_std  = scaler_raw.transform(X_test_raw_std)

# Create sequences and flatten
X_train_raw_seq, y_train_raw_seq = create_sequences(X_train_raw_std, train_data['ID'].values, y_train_raw)
X_val_raw_seq, y_val_raw_seq     = create_sequences(X_val_raw_std,   val_data['ID'].values,   y_val_raw)
X_test_raw_seq, y_test_raw_seq    = create_sequences(X_test_raw_std,  test_df['ID'].values,    y_test_raw)
X_train_raw_flat = X_train_raw_seq.reshape(X_train_raw_seq.shape[0], -1)
X_val_raw_flat   = X_val_raw_seq.reshape(X_val_raw_seq.shape[0], -1)
X_test_raw_flat  = X_test_raw_seq.reshape(X_test_raw_seq.shape[0], -1)
scaler_raw_flat = StandardScaler()
X_train_raw_norm = scaler_raw_flat.fit_transform(X_train_raw_flat)
X_val_raw_norm   = scaler_raw_flat.transform(X_val_raw_flat)
X_test_raw_norm  = scaler_raw_flat.transform(X_test_raw_flat)
np.save('fd001_X_train_raw.npy', X_train_raw_norm)
np.save('fd001_X_val_raw.npy', X_val_raw_norm)
np.save('fd001_X_test_raw.npy', X_test_raw_norm)
np.save('fd001_y_train_raw.npy', y_train_raw_seq)
np.save('fd001_y_val_raw.npy', y_val_raw_seq)
np.save('fd001_y_test_raw.npy', y_test_raw_seq)


# 8. SAVE TEST SEQUENCE IDs (for final‑point evaluation)
print("Saving test sequence IDs")
test_seq_ids = []
for engine_id in np.unique(test_df['ID'].values):
    engine_len = len(test_df[test_df['ID'] == engine_id])
    n_seq = engine_len - SEQ_LEN
    if n_seq > 0:
        test_seq_ids.extend([engine_id] * n_seq)
test_seq_ids = np.array(test_seq_ids)
np.save('fd001_test_seq_ids.npy', test_seq_ids)
print(f"Saved fd001_test_seq_ids.npy with shape {test_seq_ids.shape}")


# 9. TRAIN AND SAVE PREDICTIONS FOR EACH LEVEL
def train_and_save_predictions(X_train, y_train, X_val, y_val, X_test, y_test, name):
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, verbose=False)
    val_pred = model.predict(X_val)
    bias = np.mean(y_val - val_pred)
    test_pred = model.predict(X_test) + bias
    test_pred = gaussian_filter1d(test_pred, sigma=SIGMA, mode='nearest')
    test_pred = np.clip(test_pred, 2, RUL_CAP)
    np.save(f'fd001_predictions_{name}.npy', test_pred)
    
    # Compute and print metrics
    rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    mae = mean_absolute_error(y_test, test_pred)
    r2 = r2_score(y_test, test_pred)
    print(f"{name:12s} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
    return rmse, mae, r2

print("\nTraining and saving predictions for all levels...")
train_and_save_predictions(X_train_raw_norm, y_train_raw_seq, X_val_raw_norm, y_val_raw_seq, X_test_raw_norm, y_test_raw_seq, "A0_raw")
train_and_save_predictions(X_train_levels[3], y_train_seq, X_val_levels[3], y_val_seq, X_test_levels[3], y_test_seq, "A_stats")
train_and_save_predictions(X_train_levels[6], y_train_seq, X_val_levels[6], y_val_seq, X_test_levels[6], y_test_seq, "B_linear")
train_and_save_predictions(X_train_levels[9], y_train_seq, X_val_levels[9], y_val_seq, X_test_levels[9], y_test_seq, "C_quad")
train_and_save_predictions(X_train_levels[14], y_train_seq, X_val_levels[14], y_val_seq, X_test_levels[14], y_test_seq, "D_full")

np.save('fd001_true_values.npy', y_test_seq)

CELL 1: FD001 – Data Preparation
Removed constant sensors: ['Sensor1', 'Sensor5', 'Sensor10', 'Sensor16', 'Sensor18', 'Sensor19']
Sensors for rolling: ['Sensor2', 'Sensor3', 'Sensor4', 'Sensor6', 'Sensor7', 'Sensor8', 'Sensor9', 'Sensor11']
Total base features: 42
Train sequences: (13940, 30, 42), Val: (3691, 30, 42), Test: (10096, 30, 42)

Extracting feature matrices
Extracting 3 features per base...
Extracting 6 features per base...
Extracting 9 features per base...
Extracting 14 features per base...
Feature extraction completed in 17.68 seconds
Creating raw sensor baseline (A0)
Saving test sequence IDs
Saved fd001_test_seq_ids.npy with shape (10096,)

Training and saving predictions for all levels...
A0_raw       -> RMSE: 9.4607, MAE: 6.2740, R²: 0.7943
A_stats      -> RMSE: 7.0234, MAE: 4.4261, R²: 0.8866
B_linear     -> RMSE: 5.8744, MAE: 3.2586, R²: 0.9207
C_quad       -> RMSE: 5.8218, MAE: 3.2223, R²: 0.9221
D_full       -> RMSE: 5.7478, MAE: 3.2213, R²: 0.9241


In [2]:
# CELL 2: FD002, FD003, FD004 – Data Preparation

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.ndimage import gaussian_filter1d
import warnings
import sys
import os
import time

warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True)

print("CELL 2: FD002, FD003, FD004 – Data Preparation")


# CONFIGURATION
RANDOM_SEED = 42
RUL_CAP = 100
SEQ_LEN = 30

# Q-MetaBoost hyperparameters
xgb_params = {
    'n_estimators': 1000,
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'verbosity': 0
}

SIGMA = 0.3

# HELPER FUNCTIONS
def create_sequences(data, ids, targets, seq_len=SEQ_LEN):
    X_seq, y_seq = [], []
    for eid in np.unique(ids):
        mask = ids == eid
        edata = data[mask]
        etargets = targets[mask]
        for i in range(len(edata) - seq_len):
            X_seq.append(edata[i:i+seq_len])
            y_seq.append(etargets[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

def extract_features_vectorised(seq_data, n_features_per_base):
    n_windows, seq_len, n_base = seq_data.shape
    x = np.arange(seq_len, dtype=float)
    A_lin = np.column_stack([x, np.ones(seq_len)])
    A_quad = np.column_stack([x**2, x, np.ones(seq_len)])
    pinv_lin = np.linalg.pinv(A_lin)
    pinv_quad = np.linalg.pinv(A_quad)

    windows_by_sensor = seq_data.transpose(2, 0, 1)  # (n_base, n_windows, seq_len)
    all_feats = []
    for s_idx in range(n_base):
        w = windows_by_sensor[s_idx]                # (n_windows, seq_len)
        mean_vals = np.mean(w, axis=1)
        std_vals  = np.std(w, axis=1)
        range_vals = np.ptp(w, axis=1)
        stats_feats = np.column_stack([mean_vals, std_vals, range_vals])

        if n_features_per_base >= 6:
            lin_coeff = pinv_lin @ w.T
            slope = lin_coeff[0]
            intercept = lin_coeff[1]
            y_pred_lin = (A_lin @ lin_coeff).T
            ss_res = np.sum((w - y_pred_lin)**2, axis=1)
            ss_tot = np.sum((w - np.mean(w, axis=1, keepdims=True))**2, axis=1)
            r2 = 1 - ss_res / (ss_tot + 1e-12)
            lin_feats = np.column_stack([slope, intercept, r2])

        if n_features_per_base >= 9:
            quad_coeff = pinv_quad @ w.T
            beta2 = quad_coeff[0]
            beta1 = quad_coeff[1]
            quad_int = quad_coeff[2]
            quad_feats = np.column_stack([beta2, beta1, quad_int])

        if n_features_per_base == 14:
            momentum = np.mean(w[:, -5:], axis=1) - np.mean(w[:, :5], axis=1)
            mid = seq_len // 2
            roc_mid_start = w[:, mid] - w[:, 0]
            roc_end_mid   = w[:, -1] - w[:, mid]
            roc_avg       = (w[:, -1] - w[:, 0]) / seq_len
            roc_diff_mean = np.mean(np.diff(w, axis=1), axis=1)
            rate_feats = np.column_stack([roc_mid_start, roc_end_mid, roc_avg, roc_diff_mean])

        if n_features_per_base == 3:
            sensor_feats = stats_feats
        elif n_features_per_base == 6:
            sensor_feats = np.hstack([stats_feats, lin_feats])
        elif n_features_per_base == 9:
            sensor_feats = np.hstack([stats_feats, lin_feats, quad_feats])
        else:
            sensor_feats = np.hstack([stats_feats, lin_feats, quad_feats, momentum[:, None], rate_feats])

        all_feats.append(sensor_feats)

    return np.hstack(all_feats)

def train_and_save_predictions(X_train, y_train, X_val, y_val, X_test, y_test, ds_name, level_name):
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, verbose=False)
    val_pred = model.predict(X_val)
    bias = np.mean(y_val - val_pred)
    test_pred = model.predict(X_test) + bias
    test_pred = gaussian_filter1d(test_pred, sigma=SIGMA, mode='nearest')
    test_pred = np.clip(test_pred, 2, RUL_CAP)
    np.save(f'{ds_name}_predictions_{level_name}.npy', test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    mae = mean_absolute_error(y_test, test_pred)
    r2 = r2_score(y_test, test_pred)
    return rmse, mae, r2

# DATA PATH
IF_KAGGLE = 'kaggle' in sys.modules or os.path.exists('/kaggle/input')
if IF_KAGGLE:
    data_path = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/'
else:
    data_path = './data/CMaps/'

col_names = ['ID', 'Cycle'] + [f'Op{i}' for i in range(1,4)] + [f'Sensor{i}' for i in range(1,22)]

# PROCESS EACH DATASET
datasets = [
    {'name': 'FD002', 'train_file': 'train_FD002.txt', 'test_file': 'test_FD002.txt',
     'rul_file': 'RUL_FD002.txt', 'use_gwn': True},
    {'name': 'FD003', 'train_file': 'train_FD003.txt', 'test_file': 'test_FD003.txt',
     'rul_file': 'RUL_FD003.txt', 'use_gwn': False},
    {'name': 'FD004', 'train_file': 'train_FD004.txt', 'test_file': 'test_FD004.txt',
     'rul_file': 'RUL_FD004.txt', 'use_gwn': True}
]

for ds in datasets:
    ds_name = ds['name']
    use_gwn = ds['use_gwn']
    print(f"\nProcessing {ds_name} (GWN={use_gwn})")
    start_time = time.time()

    # Load data
    train_df = pd.read_csv(data_path + ds['train_file'], sep=r'\s+', header=None, names=col_names)
    test_df  = pd.read_csv(data_path + ds['test_file'],  sep=r'\s+', header=None, names=col_names)
    rul_df   = pd.read_csv(data_path + ds['rul_file'],   sep=r'\s+', header=None, names=['RUL'])

    # Remove constant sensors
    sensor_cols = [f'Sensor{i}' for i in range(1,22)]
    constant = [c for c in sensor_cols if train_df[c].std() < 1e-6]
    train_df.drop(columns=constant, inplace=True)
    test_df.drop(columns=constant, inplace=True)
    print(f"Removed constant sensors: {constant}")

    # Add RUL
    train_df['RUL'] = train_df.groupby('ID')['Cycle'].transform(lambda x: x.max() - x)
    train_df['RUL'] = np.minimum(train_df['RUL'], RUL_CAP)

    test_out = []
    for i, eid in enumerate(test_df['ID'].unique()):
        temp = test_df[test_df['ID'] == eid].copy()
        last_rul = rul_df.iloc[i, 0]
        temp['RUL'] = last_rul + (temp['Cycle'].max() - temp['Cycle'])
        test_out.append(temp)
    test_df = pd.concat(test_out)
    test_df['RUL'] = np.minimum(test_df['RUL'], RUL_CAP)

    # Active sensors after constant removal
    active_sensors = [c for c in sensor_cols if c not in constant]
    sensors_for_rolling = active_sensors[:8]
    print(f"Sensors for rolling: {sensors_for_rolling}")

    def add_rolling_features(df, cols):
        df = df.copy()
        for c in cols:
            df[f'{c}_mean'] = df.groupby('ID')[c].transform(
                lambda x: x.expanding().mean().shift(1).fillna(x.iloc[0]))
            df[f'{c}_std'] = df.groupby('ID')[c].transform(
                lambda x: x.expanding().std().shift(1).fillna(0))
            df[f'{c}_rate'] = df.groupby('ID')[c].transform(
                lambda x: x.diff().fillna(0))
        return df

    train_df = add_rolling_features(train_df, sensors_for_rolling)
    test_df  = add_rolling_features(test_df, sensors_for_rolling)

    op_cols = ['Op1', 'Op2', 'Op3']
    rolling_cols = []
    for c in sensors_for_rolling:
        rolling_cols.extend([f'{c}_mean', f'{c}_std', f'{c}_rate'])
    base_features = active_sensors + op_cols + rolling_cols
    print(f"Total base features: {len(base_features)}")

    # Train/val split (80/20 by engine ID)
    ids = train_df['ID'].unique()
    np.random.seed(RANDOM_SEED)
    np.random.shuffle(ids)
    split = int(0.8 * len(ids))
    train_ids = ids[:split]
    val_ids = ids[split:]

    train_data = train_df[train_df['ID'].isin(train_ids)]
    val_data   = train_df[train_df['ID'].isin(val_ids)]

    np.save(f'{ds_name.lower()}_train_ids.npy', train_ids)
    np.save(f'{ds_name.lower()}_val_ids.npy', val_ids)

    X_train_raw = train_data[base_features].values
    y_train_raw = train_data['RUL'].values
    X_val_raw   = val_data[base_features].values
    y_val_raw   = val_data['RUL'].values
    X_test_raw  = test_df[base_features].values
    y_test_raw  = test_df['RUL'].values

    # Normalisation (standard scaling)
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    X_train = imputer.fit_transform(X_train_raw)
    X_val   = imputer.transform(X_val_raw)
    X_test  = imputer.transform(X_test_raw)
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # GWN + Regime ID for FD002 and FD004
    if use_gwn:
        print("Applying Group-wise Normalization (GWN) + Regime ID")
        ops_train = train_data[op_cols].values
        ops_val   = val_data[op_cols].values
        ops_test  = test_df[op_cols].values
        kmeans = KMeans(n_clusters=6, random_state=RANDOM_SEED, n_init=10)
        regime_train = kmeans.fit_predict(ops_train)
        regime_val   = kmeans.predict(ops_val)
        regime_test  = kmeans.predict(ops_test)

        X_train_norm = np.zeros_like(X_train)
        X_val_norm   = np.zeros_like(X_val)
        X_test_norm  = np.zeros_like(X_test)

        for r in range(6):
            train_mask = regime_train == r
            val_mask   = regime_val == r
            test_mask  = regime_test == r
            if np.sum(train_mask) > 0:
                scaler_reg = StandardScaler()
                X_train_norm[train_mask] = scaler_reg.fit_transform(X_train[train_mask])
                if np.sum(val_mask) > 0:
                    X_val_norm[val_mask] = scaler_reg.transform(X_val[val_mask])
                if np.sum(test_mask) > 0:
                    X_test_norm[test_mask] = scaler_reg.transform(X_test[test_mask])
            else:
                # No training samples in this regime – use global scaling for val/test
                if np.sum(val_mask) > 0 or np.sum(test_mask) > 0:
                    global_scaler = StandardScaler()
                    global_scaler.fit(X_train)
                    if np.sum(val_mask) > 0:
                        X_val_norm[val_mask] = global_scaler.transform(X_val[val_mask])
                    if np.sum(test_mask) > 0:
                        X_test_norm[test_mask] = global_scaler.transform(X_test[test_mask])

        # Append regime ID (normalised to [0,1])
        regime_id_train = regime_train.reshape(-1, 1) / 5.0
        regime_id_val   = regime_val.reshape(-1, 1) / 5.0
        regime_id_test  = regime_test.reshape(-1, 1) / 5.0
        X_train = np.hstack([X_train_norm, regime_id_train])
        X_val   = np.hstack([X_val_norm, regime_id_val])
        X_test  = np.hstack([X_test_norm, regime_id_test])
        print(f"Added regime ID feature (normalized to [0,1]). New base feature dimension: {X_train.shape[1]}")

    else:
        print("Using standard normalization (no GWN)")

    # Create sequences
    X_train_seq, y_train_seq = create_sequences(X_train, train_data['ID'].values, y_train_raw)
    X_val_seq,   y_val_seq   = create_sequences(X_val,   val_data['ID'].values,   y_val_raw)
    X_test_seq,  y_test_seq  = create_sequences(X_test,  test_df['ID'].values,    y_test_raw)
    print(f"Train sequences: {X_train_seq.shape}, Val: {X_val_seq.shape}, Test: {X_test_seq.shape}")

    # Save test sequence IDs (for final‑point evaluation)
    test_seq_ids = []
    for engine_id in np.unique(test_df['ID'].values):
        engine_len = len(test_df[test_df['ID'] == engine_id])
        n_seq = engine_len - SEQ_LEN
        if n_seq > 0:
            test_seq_ids.extend([engine_id] * n_seq)
    test_seq_ids = np.array(test_seq_ids)
    np.save(f'{ds_name.lower()}_test_seq_ids.npy', test_seq_ids)
    print(f"Saved {ds_name.lower()}_test_seq_ids.npy with shape {test_seq_ids.shape}")

    # Feature extraction (3,6,9,14)
    print("\nExtracting feature matrices")
    X_train_levels = {}
    X_val_levels = {}
    X_test_levels = {}
    for nf in [3, 6, 9, 14]:
        print(f"Extracting {nf} features per base...")
        X_train_levels[nf] = extract_features_vectorised(X_train_seq, nf)
        X_val_levels[nf]   = extract_features_vectorised(X_val_seq, nf)
        X_test_levels[nf]  = extract_features_vectorised(X_test_seq, nf)
        scaler_feat = StandardScaler()
        X_train_levels[nf] = scaler_feat.fit_transform(X_train_levels[nf])
        X_val_levels[nf]   = scaler_feat.transform(X_val_levels[nf])
        X_test_levels[nf]  = scaler_feat.transform(X_test_levels[nf])

    # Save full level (14 features)
    np.save(f'{ds_name.lower()}_X_train_full.npy', X_train_levels[14])
    np.save(f'{ds_name.lower()}_X_val_full.npy', X_val_levels[14])
    np.save(f'{ds_name.lower()}_X_test_full.npy', X_test_levels[14])
    np.save(f'{ds_name.lower()}_y_train.npy', y_train_seq)
    np.save(f'{ds_name.lower()}_y_val.npy', y_val_seq)
    np.save(f'{ds_name.lower()}_y_test.npy', y_test_seq)

    for nf in [3, 6, 9]:
        np.save(f'{ds_name.lower()}_X_train_{nf}.npy', X_train_levels[nf])
        np.save(f'{ds_name.lower()}_X_val_{nf}.npy', X_val_levels[nf])
        np.save(f'{ds_name.lower()}_X_test_{nf}.npy', X_test_levels[nf])

    # Raw baseline (A0)
    print("Creating raw sensor baseline (A0)")
    raw_base_features = active_sensors + op_cols
    imputer_raw = SimpleImputer(strategy='mean')
    scaler_raw = StandardScaler()
    X_train_raw_std = imputer_raw.fit_transform(train_data[raw_base_features].values)
    X_val_raw_std   = imputer_raw.transform(val_data[raw_base_features].values)
    X_test_raw_std  = imputer_raw.transform(test_df[raw_base_features].values)
    X_train_raw_std = scaler_raw.fit_transform(X_train_raw_std)
    X_val_raw_std   = scaler_raw.transform(X_val_raw_std)
    X_test_raw_std  = scaler_raw.transform(X_test_raw_std)

    X_train_raw_seq, y_train_raw_seq = create_sequences(X_train_raw_std, train_data['ID'].values, y_train_raw)
    X_val_raw_seq, y_val_raw_seq     = create_sequences(X_val_raw_std,   val_data['ID'].values,   y_val_raw)
    X_test_raw_seq, y_test_raw_seq    = create_sequences(X_test_raw_std,  test_df['ID'].values,    y_test_raw)
    X_train_raw_flat = X_train_raw_seq.reshape(X_train_raw_seq.shape[0], -1)
    X_val_raw_flat   = X_val_raw_seq.reshape(X_val_raw_seq.shape[0], -1)
    X_test_raw_flat  = X_test_raw_seq.reshape(X_test_raw_seq.shape[0], -1)
    scaler_raw_flat = StandardScaler()
    X_train_raw_norm = scaler_raw_flat.fit_transform(X_train_raw_flat)
    X_val_raw_norm   = scaler_raw_flat.transform(X_val_raw_flat)
    X_test_raw_norm  = scaler_raw_flat.transform(X_test_raw_flat)
    np.save(f'{ds_name.lower()}_X_train_raw.npy', X_train_raw_norm)
    np.save(f'{ds_name.lower()}_X_val_raw.npy', X_val_raw_norm)
    np.save(f'{ds_name.lower()}_X_test_raw.npy', X_test_raw_norm)
    np.save(f'{ds_name.lower()}_y_train_raw.npy', y_train_raw_seq)
    np.save(f'{ds_name.lower()}_y_val_raw.npy', y_val_raw_seq)
    np.save(f'{ds_name.lower()}_y_test_raw.npy', y_test_raw_seq)

    # Train and save predictions
    print("Training and saving predictions for all levels...")
    results = {}
    rmse, mae, r2 = train_and_save_predictions(
        X_train_raw_norm, y_train_raw_seq,
        X_val_raw_norm, y_val_raw_seq,
        X_test_raw_norm, y_test_raw_seq,
        ds_name.lower(), "A0_raw")
    results['A0 (Raw sensors)'] = (rmse, mae, r2)

    rmse, mae, r2 = train_and_save_predictions(
        X_train_levels[3], y_train_seq,
        X_val_levels[3], y_val_seq,
        X_test_levels[3], y_test_seq,
        ds_name.lower(), "A_stats")
    results['A  (Stats)'] = (rmse, mae, r2)

    rmse, mae, r2 = train_and_save_predictions(
        X_train_levels[6], y_train_seq,
        X_val_levels[6], y_val_seq,
        X_test_levels[6], y_test_seq,
        ds_name.lower(), "B_linear")
    results['B  (+Linear)'] = (rmse, mae, r2)

    rmse, mae, r2 = train_and_save_predictions(
        X_train_levels[9], y_train_seq,
        X_val_levels[9], y_val_seq,
        X_test_levels[9], y_test_seq,
        ds_name.lower(), "C_quad")
    results['C  (+Quadratic)'] = (rmse, mae, r2)

    rmse, mae, r2 = train_and_save_predictions(
        X_train_levels[14], y_train_seq,
        X_val_levels[14], y_val_seq,
        X_test_levels[14], y_test_seq,
        ds_name.lower(), "D_full")
    results['D  (Full 14)'] = (rmse, mae, r2)

    print(f"\n{ds_name} Ablation Results (14‑feature progression)")
    print("="*60)
    print(f"{'Level':<20} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
    print("-"*60)
    for level, (rmse, mae, r2) in results.items():
        print(f"{level:<20} {rmse:8.4f} {mae:8.4f} {r2:8.4f}")
    print("="*60)

    np.save(f'{ds_name.lower()}_true_values.npy', y_test_seq)

    print(f"Time for {ds_name}: {time.time()-start_time:.2f} seconds")
    print(f"{ds_name} processing complete.")

CELL 2: FD002, FD003, FD004 – Data Preparation

Processing FD002 (GWN=True)
Removed constant sensors: []
Sensors for rolling: ['Sensor1', 'Sensor2', 'Sensor3', 'Sensor4', 'Sensor5', 'Sensor6', 'Sensor7', 'Sensor8']
Total base features: 48
Applying Group-wise Normalization (GWN) + Regime ID
Added regime ID feature (normalized to [0,1]). New base feature dimension: 49
Train sequences: (36649, 30, 49), Val: (9310, 30, 49), Test: (26252, 30, 49)
Saved fd002_test_seq_ids.npy with shape (26252,)

Extracting feature matrices
Extracting 3 features per base...
Extracting 6 features per base...
Extracting 9 features per base...
Extracting 14 features per base...
Creating raw sensor baseline (A0)
Training and saving predictions for all levels...

FD002 Ablation Results (14‑feature progression)
Level                    RMSE      MAE       R²
------------------------------------------------------------
A0 (Raw sensors)      12.2289   8.8414   0.6911
A  (Stats)            13.1874   8.3911   0.6408
B

In [3]:
# CELL 3: Statistical Validation, Benchmarking, and Final Tables

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
from scipy.ndimage import gaussian_filter1d
import time
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True)

# CONFIGURATION
RANDOM_SEED = 42
RUL_CAP = 100
SIGMA = 0.3

# MetaBoost-RC hyperparameters (same as used for training)
xgb_params = {
    'n_estimators': 1000,
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'verbosity': 0
}


# HELPER FUNCTIONS
def nasa_score(errors):
    """Compute NASA S-score from prediction errors (ŷ - y)."""
    errors = np.array(errors)
    early = errors[errors < 0]
    late  = errors[errors >= 0]
    score = np.sum(np.exp(-early/13) - 1) + np.sum(np.exp(late/10) - 1)
    return score

def final_point_metrics(predictions, true_values, seq_ids):
    """Compute final‑point RMSE, R², NASA S‑score, and avg penalty per engine."""
    unique_ids = np.unique(seq_ids)
    final_pred = []
    final_true = []
    for eid in unique_ids:
        mask = seq_ids == eid
        if np.sum(mask) == 0:
            continue
        last_idx = np.where(mask)[0][-1]
        final_pred.append(predictions[last_idx])
        final_true.append(true_values[last_idx])
    final_pred = np.array(final_pred)
    final_true = np.array(final_true)
    rmse = np.sqrt(mean_squared_error(final_true, final_pred))
    r2 = r2_score(final_true, final_pred)
    errors = final_pred - final_true
    score = nasa_score(errors)
    avg_penalty = score / len(unique_ids)
    return rmse, r2, score, avg_penalty

def critical_rmse(predictions, true_values, threshold=30):
    """RMSE for samples where true RUL < threshold."""
    mask = true_values < threshold
    if np.sum(mask) == 0:
        return np.nan
    return np.sqrt(mean_squared_error(true_values[mask], predictions[mask]))

# 1. LOAD SAVED DATA
datasets = ['fd001', 'fd002', 'fd003', 'fd004']
all_true = {}
all_pred_levels = {}  # levels: A0_raw, A_stats, B_linear, C_quad, D_full
all_seq_ids = {}      # for final‑point metrics (optional)

print("Loading saved predictions and true values")
for ds in datasets:
    # True values (must exist)
    true_path = f'{ds}_true_values.npy'
    if os.path.exists(true_path):
        all_true[ds] = np.load(true_path)
    else:
        print(f"Warning: {true_path} not found. Skipping {ds}.")
        continue
    
    # Predictions for each level
    levels = ['A0_raw', 'A_stats', 'B_linear', 'C_quad', 'D_full']
    all_pred_levels[ds] = {}
    for level in levels:
        pred_path = f'{ds}_predictions_{level}.npy'
        if os.path.exists(pred_path):
            all_pred_levels[ds][level] = np.load(pred_path)
        else:
            print(f"Warning: {pred_path} not found.")
    
    # Load test sequence IDs for final‑point evaluation (if available)
    seq_path = f'{ds}_test_seq_ids.npy'
    if not os.path.exists(seq_path):
        alt_path = f'test_seq_ids_{ds.upper()}.npy'
        if os.path.exists(alt_path):
            seq_path = alt_path
    if os.path.exists(seq_path):
        all_seq_ids[ds] = np.load(seq_path)
    else:
        print(f"Info: Sequence IDs for {ds} not found. Final‑point metrics will be skipped for this dataset.")

print("Data loading complete.\n")

# 2. ABLATION TABLE (RMSE, MAE, R²)
print("ABLATION TABLE (14‑feature progression)")

ablation_rows = []
for ds in datasets:
    if ds not in all_true or all_true[ds] is None:
        continue
    y_true = all_true[ds]
    for level_name, pred in all_pred_levels[ds].items():
        rmse = np.sqrt(mean_squared_error(y_true, pred))
        mae = mean_absolute_error(y_true, pred)
        r2 = r2_score(y_true, pred)
        ablation_rows.append({
            'Dataset': ds.upper(),
            'Level': level_name,
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2
        })

ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))
ablation_df.to_csv('ablation_table.csv', index=False)

# 3. STATISTICAL SIGNIFICANCE (Paired t-test, Cohen's d)
print("STATISTICAL SIGNIFICANCE (Full model vs Raw baseline)")

stats_results = []
for ds in datasets:
    if ds not in all_true or all_true[ds] is None:
        continue
    if 'A0_raw' not in all_pred_levels[ds] or 'D_full' not in all_pred_levels[ds]:
        continue
    y_true = all_true[ds]
    pred_raw = all_pred_levels[ds]['A0_raw']
    pred_full = all_pred_levels[ds]['D_full']
    
    abs_err_raw = np.abs(y_true - pred_raw)
    abs_err_full = np.abs(y_true - pred_full)
    
    t_stat, p_val = stats.ttest_rel(abs_err_raw, abs_err_full)
    diff = abs_err_raw - abs_err_full
    d = np.mean(diff) / np.std(diff, ddof=1)
    
    # Bootstrap CI for full model RMSE
    n_bootstrap = 5000
    np.random.seed(RANDOM_SEED)
    boot_rmse = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        boot_rmse.append(np.sqrt(mean_squared_error(y_true[idx], pred_full[idx])))
    ci_lower, ci_upper = np.percentile(boot_rmse, [2.5, 97.5])
    
    stats_results.append({
        'Dataset': ds.upper(),
        't_statistic': t_stat,
        'p_value': p_val,
        'Cohen_d': d,
        'Full_RMSE': np.sqrt(mean_squared_error(y_true, pred_full)),
        'CI_95_lower': ci_lower,
        'CI_95_upper': ci_upper
    })

stats_df = pd.DataFrame(stats_results)
print(stats_df.to_string(index=False))
stats_df.to_csv('statistical_significance.csv', index=False)


# 4. STABILITY TEST (5 random seeds) on full model
print("STABILITY TEST (5 seeds) – Full model")

stability_results = []
seeds = [42, 1, 10, 100, 2025]

for ds in datasets:
    if ds not in all_true or all_true[ds] is None:
        continue
    # Load feature matrices
    X_train_path = f'{ds}_X_train_full.npy'
    X_val_path   = f'{ds}_X_val_full.npy'
    X_test_path  = f'{ds}_X_test_full.npy'
    y_train_path = f'{ds}_y_train.npy'
    y_val_path   = f'{ds}_y_val.npy'
    y_test_path  = f'{ds}_y_test.npy'
    
    if not all(os.path.exists(p) for p in [X_train_path, X_val_path, X_test_path, y_train_path, y_val_path, y_test_path]):
        print(f"Feature matrices for {ds} not found. Skipping stability test.")
        continue
    
    X_train = np.load(X_train_path)
    X_val   = np.load(X_val_path)
    X_test  = np.load(X_test_path)
    y_train = np.load(y_train_path)
    y_val   = np.load(y_val_path)
    y_test  = np.load(y_test_path)
    
    rmse_list = []
    for seed in seeds:
        params = xgb_params.copy()
        params['random_state'] = seed
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, verbose=False)
        val_pred = model.predict(X_val)
        bias = np.mean(y_val - val_pred)
        test_pred = model.predict(X_test) + bias
        test_pred = gaussian_filter1d(test_pred, sigma=SIGMA, mode='nearest')
        test_pred = np.clip(test_pred, 2, RUL_CAP)
        rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        rmse_list.append(rmse)
    
    mean_rmse = np.mean(rmse_list)
    std_rmse = np.std(rmse_list)
    stability_results.append({
        'Dataset': ds.upper(),
        'Mean_RMSE': mean_rmse,
        'Std_RMSE': std_rmse,
        'CV (%)': (std_rmse / mean_rmse) * 100 if mean_rmse > 0 else 0
    })
    print(f"{ds.upper()}: Mean RMSE = {mean_rmse:.4f} ± {std_rmse:.4f} (CV={stability_results[-1]['CV (%)']:.2f}%)")

stability_df = pd.DataFrame(stability_results)
stability_df.to_csv('stability_test.csv', index=False)

# 5. FEATURE IMPORTANCE (FD001) – 14 features per base
print("FEATURE IMPORTANCE (FD001)")

# Load full feature matrix for FD001
X_train_fd001 = np.load('fd001_X_train_full.npy')
y_train_fd001 = np.load('fd001_y_train.npy')

model = xgb.XGBRegressor(**xgb_params)
model.fit(X_train_fd001, y_train_fd001, verbose=False)
importance = model.feature_importances_

n_base = X_train_fd001.shape[1] // 14
print(f"Number of base features: {n_base}")

categories = {
    'Statistics': 0,
    'Linear Trend': 0,
    'Quadratic': 0,
    'Momentum': 0,
    'Rate of Change': 0
}

for i, imp in enumerate(importance):
    feat_in_base = i % 14
    if feat_in_base < 3:
        categories['Statistics'] += imp
    elif feat_in_base < 6:
        categories['Linear Trend'] += imp
    elif feat_in_base < 9:
        categories['Quadratic'] += imp
    elif feat_in_base == 9:
        categories['Momentum'] += imp
    else:
        categories['Rate of Change'] += imp

total = sum(categories.values())
for cat in categories:
    categories[cat] = categories[cat] / total * 100

print("Feature importance by category (%):")
for cat, val in sorted(categories.items(), key=lambda x: x[1], reverse=True):
    print(f"{cat}: {val:.1f}%")

imp_df = pd.DataFrame(list(categories.items()), columns=['Category', 'Importance_%'])
imp_df.to_csv('feature_importance_categories.csv', index=False)

# 6. FINAL-POINT METRICS (NASA S-score)
print("FINAL-POINT METRICS (NASA S-score)")

final_metrics = []
for ds in datasets:
    if ds not in all_true or all_true[ds] is None:
        continue
    if 'D_full' not in all_pred_levels[ds]:
        continue
    y_test = all_true[ds]
    pred_full = all_pred_levels[ds]['D_full']
    
    if ds not in all_seq_ids:
        print(f"Sequence IDs for {ds} not found. Skipping final-point metrics.")
        continue
    
    seq_ids = all_seq_ids[ds]
    rmse_final, r2_final, score, avg_penalty = final_point_metrics(pred_full, y_test, seq_ids)
    final_metrics.append({
        'Dataset': ds.upper(),
        'Final_RMSE': rmse_final,
        'Final_R²': r2_final,
        'NASA_S_Score': score,
        'Avg_Penalty_per_Engine': avg_penalty
    })

if final_metrics:
    final_df = pd.DataFrame(final_metrics)
    print(final_df.to_string(index=False))
    final_df.to_csv('final_point_metrics.csv', index=False)
else:
    print("No final-point metrics computed (missing sequence IDs).")

# 7. CRITICAL REGION RMSE (RUL < 30)
print("CRITICAL REGION RMSE (RUL < 30)")

critical_results = []
for ds in datasets:
    if ds not in all_true or all_true[ds] is None:
        continue
    if 'D_full' not in all_pred_levels[ds]:
        continue
    y_test = all_true[ds]
    pred_full = all_pred_levels[ds]['D_full']
    critical_rmse_val = critical_rmse(pred_full, y_test, threshold=30)
    global_rmse_val = np.sqrt(mean_squared_error(y_test, pred_full))
    error_reduction = (global_rmse_val - critical_rmse_val) / global_rmse_val * 100 if global_rmse_val > 0 else 0
    # Bootstrap CI for critical RMSE
    n_bootstrap = 5000
    np.random.seed(RANDOM_SEED)
    boot_crit = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)
        y_boot = y_test[idx]
        pred_boot = pred_full[idx]
        mask = y_boot < 30
        if np.sum(mask) > 0:
            boot_crit.append(np.sqrt(mean_squared_error(y_boot[mask], pred_boot[mask])))
    if len(boot_crit) > 0:
        ci_lower, ci_upper = np.percentile(boot_crit, [2.5, 97.5])
    else:
        ci_lower, ci_upper = np.nan, np.nan
    
    critical_results.append({
        'Dataset': ds.upper(),
        'Global_RMSE': global_rmse_val,
        'Critical_RMSE': critical_rmse_val,
        'Error_Reduction_%': error_reduction,
        '95%_CI_lower': ci_lower,
        '95%_CI_upper': ci_upper,
        'Sample_Size': np.sum(y_test < 30)
    })

critical_df = pd.DataFrame(critical_results)
print(critical_df.to_string(index=False))
critical_df.to_csv('critical_region_rmse.csv', index=False)

Loading saved predictions and true values
Data loading complete.

ABLATION TABLE (14‑feature progression)
Dataset    Level      RMSE      MAE       R²
  FD001   A0_raw  9.460699 6.274001 0.794265
  FD001  A_stats  7.023369 4.426107 0.886616
  FD001 B_linear  5.874417 3.258640 0.920678
  FD001   C_quad  5.821811 3.222256 0.922093
  FD001   D_full  5.747802 3.221338 0.924061
  FD002   A0_raw 12.228889 8.841401 0.691135
  FD002  A_stats 13.187380 8.391128 0.640820
  FD002 B_linear  9.946334 6.528772 0.795675
  FD002   C_quad  9.954779 6.557823 0.795328
  FD002   D_full  9.853993 6.513948 0.799452
  FD003   A0_raw  7.562832 5.141932 0.821832
  FD003  A_stats  6.446358 3.836447 0.870554
  FD003 B_linear  5.786362 3.492933 0.895703
  FD003   C_quad  5.661762 3.375518 0.900146
  FD003   D_full  5.553355 3.179088 0.903933
  FD004   A0_raw 10.712895 5.972979 0.671837
  FD004  A_stats 11.155847 6.163702 0.644138
  FD004 B_linear  9.004872 4.709448 0.768137
  FD004   C_quad  8.965494 4.636062 0.7